# Notebook 04: Evaluation and Interpretation

**Project:** GNN-Based Battery Voltage Predictor  

---

## Overview

This notebook evaluates all three trained models (RF, CGCNN, M3GNet) on the
held-out test set and produces the following analyses:

1. **Benchmark table**: MAE, RMSE, R-squared for all models side by side
2. **Parity plots**: predicted vs actual voltage, colored by chemistry family
3. **Error analysis by chemistry**: which families are hardest to predict?
4. **Feature importance** (Random Forest): top MAGPIE features driving prediction
5. **Gradient-based sensitivity** (CGCNN): input gradient magnitude per node feature
6. **Model comparison chart**: grouped bar plot for the results section

All figures are saved to `results/` with publication settings.

In [ ]:
import sys
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import VoltageGraphDataset
from src.models import build_cgcnn, CrystalTransformer
from src.train import predict, make_loaders
from src.evaluate import (
    compute_metrics, print_metrics, parity_plot,
    error_by_chemistry, model_comparison_chart, PALETTE
)
from src.utils import set_seed

set_seed(42)

DATA_DIR    = project_root / 'data'
MODELS_DIR  = project_root / 'models'
RESULTS_DIR = project_root / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
# Load test set
test_ds  = VoltageGraphDataset.from_processed(str(DATA_DIR / 'test_graphs.pkl'))
_, _, test_loader = make_loaders(
    test_ds, test_ds, test_ds, batch_size=128, num_workers=4
)
# Note: only test_loader is used below

# Load matminer features
with open(DATA_DIR / 'matminer_filtered.pkl', 'rb') as f:
    mm = pickle.load(f)
X_test, y_test_mm = mm['X_test'], mm['y_test']
feat_names = mm['feature_names']

# Collect metadata for test set
with open(DATA_DIR / 'splits.pkl', 'rb') as f:
    splits = pickle.load(f)
test_entries = splits['test']

# Chemistry family labels aligned with graph dataset order
# (VoltageGraphDataset preserves entry order for valid structures)
test_chemistry = [e['chemistry_family'] for e in test_entries
                  if e.get('charged_structure') or e.get('discharged_structure')]

print(f'Test entries: {len(test_entries)}')
print(f'Test graph dataset: {len(test_ds)}')
print(f'Chemistry labels: {len(test_chemistry)}')

## 1. Load Trained Models and Generate Test Predictions

In [ ]:
# Random Forest predictions
with open(MODELS_DIR / 'rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

# Align matminer test set with graph test set (entries that produced valid graphs)
# Use a length match heuristic; for production code, track IDs explicitly
n_test_graphs = len(test_ds)
y_test_rf = y_test_mm[:n_test_graphs]
X_test_aligned = X_test[:n_test_graphs]

rf_pred = rf_model.predict(X_test_aligned)
rf_metrics = compute_metrics(y_test_rf, rf_pred)
print_metrics('Random Forest', rf_metrics)

In [ ]:
# CGCNN predictions
cgcnn = build_cgcnn(node_dim=9, edge_dim=64, hidden_dim=128, n_conv=4, dropout=0.1)
cgcnn.load_state_dict(torch.load(MODELS_DIR / 'cgcnn_best.pt', map_location=device))
cgcnn = cgcnn.to(device)

cgcnn_pred, cgcnn_true = predict(cgcnn, test_loader, device)
cgcnn_metrics = compute_metrics(cgcnn_true, cgcnn_pred)
print_metrics('CGCNN', cgcnn_metrics)

In [ ]:
# CrystalTransformer predictions
ct_model = CrystalTransformer(node_dim=9, edge_dim=64, hidden_dim=128,
                               n_conv=4, heads=4, dropout=0.1)
ct_model.load_state_dict(torch.load(MODELS_DIR / 'transformer_best.pt', map_location=device))
ct_model = ct_model.to(device)

ct_pred, ct_true = predict(ct_model, test_loader, device)
ct_metrics = compute_metrics(ct_true, ct_pred)
print_metrics('CrystalTransformer', ct_metrics)


## 2. Benchmark Table

In [ ]:
all_metrics = {
    'Random Forest':      rf_metrics,
    'CGCNN':              cgcnn_metrics,
    'CrystalTransformer': ct_metrics,
}

benchmark_df = pd.DataFrame({
    'Model':      list(all_metrics.keys()),
    'MAE (V)':    [v['MAE']  for v in all_metrics.values()],
    'RMSE (V)':   [v['RMSE'] for v in all_metrics.values()],
    'R-squared':  [v['R2']   for v in all_metrics.values()],
})
benchmark_df = benchmark_df.sort_values('MAE (V)')
benchmark_df[['MAE (V)', 'RMSE (V)', 'R-squared']] = (
    benchmark_df[['MAE (V)', 'RMSE (V)', 'R-squared']].round(4)
)
print(benchmark_df.to_string(index=False))

benchmark_df.to_csv(RESULTS_DIR / 'benchmark_table.csv', index=False)
print('\nSaved benchmark table.')


## 3. Parity Plots

We generate a 3-panel parity plot with one panel per model.
Points are colored by chemistry family.

In [ ]:
# Chemistry labels aligned with the graph test set
test_chemistry = [e['chemistry_family'] for e in splits['test']]

# RF aligned to matminer test set (same order)
rf_chem   = test_chemistry[:len(rf_pred)]
cgcnn_chem = test_chemistry[:len(cgcnn_pred)]
ct_chem    = test_chemistry[:len(ct_pred)]

for model_name, y_true, y_pred_arr, chem_labels in [
    ('Random Forest',      y_test_rf,    rf_pred,   rf_chem),
    ('CGCNN',              cgcnn_true,   cgcnn_pred, cgcnn_chem),
    ('CrystalTransformer', ct_true,      ct_pred,    ct_chem),
]:
    fig = parity_plot(
        y_true, y_pred_arr,
        labels=chem_labels,
        title=f'{model_name}: Predicted vs Actual Voltage',
        model_name=model_name.lower().replace(' ', '_'),
        save_path=str(RESULTS_DIR / f'fig04_parity_{model_name.lower().replace(" ", "_")}.png'),
    )
    plt.close(fig)

print('Individual parity plots saved.')


In [ ]:
# Combined 3-panel parity figure
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (model_name, y_true, y_pred_arr, chem_labels, metrics) in zip(axes, [
    ('RF',               y_test_rf,  rf_pred,   rf_chem,   rf_metrics),
    ('CGCNN',            cgcnn_true, cgcnn_pred, cgcnn_chem, cgcnn_metrics),
    ('CrystalTransformer', ct_true,  ct_pred,    ct_chem,  ct_metrics),
]):
    unique_labels = sorted(set(chem_labels))
    for lbl in unique_labels:
        mask = np.array(chem_labels) == lbl
        ax.scatter(y_true[mask], y_pred_arr[mask],
                   s=10, alpha=0.5, label=lbl,
                   color=PALETTE.get(lbl, '#607D8B'), linewidths=0)
    vmin = min(y_true.min(), y_pred_arr.min()) - 0.2
    vmax = max(y_true.max(), y_pred_arr.max()) + 0.2
    ax.plot([vmin, vmax], [vmin, vmax], 'k--', lw=1.0, alpha=0.5)
    ann = f"MAE={metrics['MAE']:.3f} V\nR$^2$={metrics['R2']:.3f}"
    ax.text(0.97, 0.04, ann, transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='grey', alpha=0.8))
    ax.set_xlabel('DFT Voltage (V vs Li/Li+)')
    ax.set_ylabel('Predicted Voltage (V)')
    ax.set_title(model_name)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_aspect('equal')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].legend(title='Chemistry', fontsize=7.5, loc='upper left',
               frameon=True, framealpha=0.8)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig04_parity_combined.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved: fig04_parity_combined.png')


## 4. Error Analysis by Chemistry Family

In [ ]:
# Error breakdown by chemistry family
fig = error_by_chemistry(
    cgcnn_true, cgcnn_pred, cgcnn_chem,
    save_path=str(RESULTS_DIR / 'fig04_error_by_chemistry_cgcnn.png')
)
plt.close(fig)

fig = error_by_chemistry(
    ct_true, ct_pred, ct_chem,
    save_path=str(RESULTS_DIR / 'fig04_error_by_chemistry_transformer.png')
)
plt.close(fig)

print('Error by chemistry figures saved.')


In [ ]:
# Residual plot: error vs actual voltage (CGCNN)
residuals = cgcnn_pred - cgcnn_true

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
for lbl in sorted(set(cgcnn_chem)):
    mask = np.array(cgcnn_chem) == lbl
    ax.scatter(cgcnn_true[mask], residuals[mask],
               s=10, alpha=0.5, label=lbl,
               color=PALETTE.get(lbl, '#607D8B'), linewidths=0)
ax.axhline(0, color='black', lw=1.0, linestyle='--', alpha=0.5)
ax.set_xlabel('DFT Voltage (V vs Li/Li+)')
ax.set_ylabel('Residual: Pred minus True (V)')
ax.set_title('CGCNN Residuals vs Actual Voltage')
ax.legend(title='Chemistry', fontsize=7.5, frameon=True, framealpha=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax2 = axes[1]
ax2.hist(residuals, bins=50, color='#2196F3', alpha=0.8, edgecolor='none')
ax2.axvline(0, color='black', lw=1.0, linestyle='--', alpha=0.5)
ax2.axvline(residuals.mean(), color='#F44336', lw=1.5, linestyle='--',
            label=f'Mean bias: {residuals.mean():.3f} V')
ax2.set_xlabel('Residual (V)')
ax2.set_ylabel('Count')
ax2.set_title('CGCNN Residual Distribution')
ax2.legend(frameon=False)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig04_cgcnn_residuals.png')
plt.show()

## 5. Feature Importance (Random Forest)

Scikit-learn's mean-decrease-in-impurity importance is plotted for the top 20
features. This reveals which elemental properties most strongly influence the
random forest voltage prediction.

In [ ]:
importances = rf_model.feature_importances_
std = np.std([tree.feature_importances_ for tree in rf_model.estimators_], axis=0)
top_n = 20
top_idx = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#4CAF50' if i < 5 else '#2196F3' for i in range(top_n)]
ax.barh(range(top_n), importances[top_idx[::-1]],
        xerr=std[top_idx[::-1]], color=colors[::-1],
        alpha=0.85, edgecolor='none', error_kw=dict(elinewidth=0.8, capsize=2))
ax.set_yticks(range(top_n))
ax.set_yticklabels([feat_names[i] for i in top_idx[::-1]], fontsize=8.5)
ax.set_xlabel('Mean Decrease in Impurity (Normalized)')
ax.set_title('Random Forest Feature Importance: Top 20 Descriptors')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig04_rf_feature_importance.png')
plt.show()
print('Saved: fig04_rf_feature_importance.png')

## 6. CGCNN Gradient Sensitivity

We compute the mean absolute gradient of the voltage prediction with respect
to each node feature dimension, averaged across all test structures.
This serves as a simple attribution/saliency measure indicating which
atom features the network attends to most strongly.

In [ ]:
cgcnn.eval()
feat_dim = 9
feat_labels = [
    'Atomic Z (norm)', 'Electronegativity', 'Ionic Radius',
    'Group', 'Period', 'Is Metal',
    'Is Trans. Metal', 'Is Alkali', 'Is Alkaline'
]

grad_accumulator = np.zeros(feat_dim)
n_batches = 0

for batch in test_loader:
    batch = batch.to(device)
    batch.x.requires_grad_(True)
    pred = cgcnn(batch)
    pred.sum().backward()
    grads = batch.x.grad.abs().mean(dim=0).detach().cpu().numpy()
    grad_accumulator += grads
    n_batches += 1
    if n_batches >= 20:  # Use first 20 batches for speed
        break

mean_grads = grad_accumulator / n_batches
norm_grads = mean_grads / mean_grads.sum()

fig, ax = plt.subplots(figsize=(7, 3.5))
colors_grad = plt.cm.viridis(np.linspace(0.2, 0.8, feat_dim))
ax.bar(range(feat_dim), norm_grads, color=colors_grad, alpha=0.85, edgecolor='none')
ax.set_xticks(range(feat_dim))
ax.set_xticklabels(feat_labels, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Normalized Mean Gradient Magnitude')
ax.set_title('CGCNN Node Feature Sensitivity (Gradient Attribution)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig04_cgcnn_gradient_sensitivity.png')
plt.show()
print('Saved: fig04_cgcnn_gradient_sensitivity.png')

## 7. Model Comparison Chart

In [ ]:
fig = model_comparison_chart(
    {
        'RF':               rf_metrics,
        'CGCNN':            cgcnn_metrics,
        'CrystalTransformer': ct_metrics,
    },
    save_path=str(RESULTS_DIR / 'fig04_model_comparison.png')
)
plt.close(fig)
print('Saved: fig04_model_comparison.png')


In [ ]:
# Determine best model
best_name = min(all_metrics, key=lambda k: all_metrics[k]['MAE'])
best_m = all_metrics[best_name]

print('Evaluation complete. Key findings:')
for name, m in sorted(all_metrics.items(), key=lambda kv: kv[1]['MAE']):
    marker = '  <-- best' if name == best_name else ''
    print(f"  {name:22s}  MAE: {m['MAE']:.4f} V  RMSE: {m['RMSE']:.4f} V  R2: {m['R2']:.4f}{marker}")
print(f'\nAll figures saved to {RESULTS_DIR}')
print('Proceed to 05_screening.ipynb')
